# Experiment 1 — Pretrained vs From Scratch

## Dataset Preparation

In [1]:
import pandas as pd
import numpy as np
import torch
import random

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader

SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 4

MODEL_NAME = "distilbert-base-uncased"
# Hoặc MODEL_NAME = "bert-base-uncased"

DATA_PATH = "/home/vitquay1708/Study_Space/NLP/lab4/imdb/IMDB Dataset.csv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.head())
print(df["sentiment"].value_counts())

/home/vitquay1708/miniconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [2]:
label_map = {
    "negative": 0,
    "positive": 1
}

df = df[["review", "sentiment"]].dropna().copy()
df["label"] = df["sentiment"].map(label_map)

assert df["label"].notna().all(), "Unknown sentiment label"

df = df[["review", "label"]].reset_index(drop=True)
df["label"] = df["label"].astype(int)

print(df.head())

                                              review  label
0  One of the other reviewers has mentioned that ...      1
1  A wonderful little production. <br /><br />The...      1
2  I thought this was a wonderful way to spend ti...      1
3  Basically there's a family where a little boy ...      0
4  Petter Mattei's "Love in the Time of Money" is...      1


In [3]:
# Check duplicates
print("Before cleaning:", len(df))

duplicate_count = df.duplicated(subset=["review"]).sum()
print("Duplicate reviews:", duplicate_count)

# Kiểm tra các review giống nhau nhưng có label khác nhau
conflicting_labels = (
    df.groupby("review")["label"]
      .nunique()
)

conflicting_reviews = conflicting_labels[
    conflicting_labels > 1
].index

print("Conflicting reviews:", len(conflicting_reviews))

# Loại bỏ các review có nhãn mâu thuẫn
df = df[~df["review"].isin(conflicting_reviews)]

# Xóa duplicate dựa trên nội dung review
df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)

print("After cleaning:", len(df))

Before cleaning: 50000
Duplicate reviews: 418
Conflicting reviews: 0
After cleaning: 49582


In [4]:
# 50% train+validation, 50% test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.5,
    random_state=SEED,
    stratify=df["label"]
)

# 80% train, 20% validation trong train_val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_val_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 19832
Validation: 4959
Test: 24791


In [5]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(
 train_df, preserve_index=False
    ),
    "validation": Dataset.from_pandas(
 val_df, preserve_index=False
    ),
    "test": Dataset.from_pandas(
 test_df, preserve_index=False
    )
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 19832
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 4959
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 24791
    })
})


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
 batch["review"],
 truncation=True,
 max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["review"]
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

Map: 100%|██████████| 24791/24791 [00:08<00:00, 2924.48 examples/s]


In [7]:
train_dataset = tokenized_dataset["train"]
val_dataset = tokenized_dataset["validation"]
test_dataset = tokenized_dataset["test"]

def make_train_loader():
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
 train_dataset,
 batch_size=BATCH_SIZE,
 shuffle=True,
 generator=generator,
 collate_fn=data_collator
    )
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

train_loader = make_train_loader()
batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([4, 128])
attention_mask torch.Size([4, 128])
labels torch.Size([4])


In [8]:
for name, data in dataset.items():
    labels = np.array(data["label"])

    print(f"\n{name}")
    print("Total samples:", len(data))
    print("Negative:", (labels == 0).sum())
    print("Positive:", (labels == 1).sum())

# Kiểm tra không có sample trùng giữa các split
train_ids = set(train_df.index)

train_texts = set(train_df["review"])
val_texts = set(val_df["review"])
test_texts = set(test_df["review"])

print("Train-Val overlap:", len(train_texts & val_texts))
print("Train-Test overlap:", len(train_texts & test_texts))
print("Val-Test overlap:", len(val_texts & test_texts))


train
Total samples: 19832
Negative: 9879
Positive: 9953

validation
Total samples: 4959
Negative: 2470
Positive: 2489

test
Total samples: 24791
Negative: 12349
Positive: 12442
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


## Model preparation

In [9]:
import torch
import gc

from transformers import AutoConfig, AutoModelForSequenceClassification, set_seed


NUM_LABELS = 2

ID2LABEL = {
    0: "negative",
    1: "positive"
}

LABEL2ID = {
    "negative": 0,
    "positive": 1
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

Model: distilbert-base-uncased
Device: cuda


In [10]:
set_seed(SEED)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

model_a = AutoModelForSequenceClassification.from_config(config)
model_b= AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)



Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1810.74it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
      "total": total_params,
      "trainable": trainable_params
    }

In [12]:
params_a = count_parameters(model_a)

print("Model A — From Scratch")
print(f"Total parameters: {params_a['total']:,}")
print(f"Trainable parameters: {params_a['trainable']:,}")

params_b = count_parameters(model_b)

print("\nModel B — Pretrained")
print(f"Total parameters: {params_b['total']:,}")
print(f"Trainable parameters: {params_b['trainable']:,}")

# Verify architecture parameter counts
assert params_a == params_b, "Parameter counts are different!"

# Verify full fine-tuning
assert params_b["total"] == params_b["trainable"]

print("\nBoth models have identical parameter counts.")


Model A — From Scratch
Total parameters: 66,955,010
Trainable parameters: 66,955,010

Model B — Pretrained
Total parameters: 66,955,010
Trainable parameters: 66,955,010

Both models have identical parameter counts.


## Training and Validation

In [13]:
import os
import gc
import math
import time
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, f1_score
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# Training configuration
EPOCHS = 3
MICRO_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

USE_AMP = torch.cuda.is_available()
USE_GRADIENT_CHECKPOINTING = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = Path("./results/experiment1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Effective batch size:",
      MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
Model: distilbert-base-uncased
Effective batch size: 32
GPU: NVIDIA GeForce GTX 1650


In [14]:
def create_train_loader(dataset, seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=data_collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )


def create_val_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

In [15]:
# True: test the pipeline with 128 training samples
# False: train on the entire training dataset
SMOKE_TEST = False

if SMOKE_TEST:
    experiment_train_dataset = Subset(
        train_dataset,
        range(min(128, len(train_dataset)))
    )
    experiment_val_dataset = Subset(
        val_dataset,
        range(min(64, len(val_dataset)))
    )
else:
    experiment_train_dataset = train_dataset
    experiment_val_dataset = val_dataset

print("Train samples:", len(experiment_train_dataset))
print("Validation samples:", len(experiment_val_dataset))

Train samples: 19832
Validation samples: 4959


In [16]:
@torch.inference_mode()
def evaluate_model(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_predictions = []
    all_labels = []

    for batch in data_loader:
        batch = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }

        with autocast_context():
            outputs = model(**batch)

        labels = batch["labels"]
        predictions = outputs.logits.argmax(dim=-1)

        batch_size = labels.size(0)

        total_loss += outputs.loss.item() * batch_size
        total_samples += batch_size

        all_predictions.extend(
            predictions.cpu().tolist()
        )
        all_labels.extend(
            labels.cpu().tolist()
        )

    accuracy = accuracy_score(
        all_labels, all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    positive_f1 = f1_score(
        all_labels,
        all_predictions,
        pos_label=1,
        zero_division=0
    )

    return {
        "val_loss": total_loss / total_samples,
        "val_accuracy": accuracy,
        "val_macro_f1": macro_f1,
        "val_positive_f1": positive_f1
    }


def autocast_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )
    return nullcontext()

In [17]:
from transformers import set_seed
def train_one_model(model_name, model):
    # 1. Reproducibility
    set_seed(SEED)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    if USE_GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )

    model.to(DEVICE)

    # 3. DataLoaders
    train_dl = create_train_loader(
        experiment_train_dataset,
        seed=SEED
    )

    val_dl = create_val_loader(
        experiment_val_dataset
    )

    # 4. Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # 5. Learning rate scheduler
    steps_per_epoch = math.ceil(
        len(train_dl) / GRAD_ACCUM_STEPS
    )

    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # 6. Automatic mixed precision
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    # 7. Training history
    history = []

    best_f1 = -1.0
    best_epoch = None
    total_training_time = 0.0

    total_params = sum(
        p.numel() for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    suffix = "_smoke" if SMOKE_TEST else ""
    checkpoint_path = (
        OUTPUT_DIR / f"{model_name}{suffix}_best.pt"
    )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    print(f"\n{'=' * 60}")
    print(f"Training: {model_name}")
    print(f"Parameters: {total_params:,}")
    print(f"Epochs: {EPOCHS}")
    print(f"Optimizer steps: {total_steps}")
    print(f"{'=' * 60}")

    # 8. Training epochs
    for epoch in range(1, EPOCHS + 1):
        model.train()

        epoch_loss_sum = torch.zeros(
            (), device=DEVICE
        )
        epoch_samples = 0

        optimizer.zero_grad(set_to_none=True)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        progress_bar = tqdm(
            enumerate(train_dl),
            total=len(train_dl),
            desc=f"{model_name} | Epoch {epoch}/{EPOCHS}"
        )

        for batch_idx, batch in progress_bar:

            batch = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
            }

            # Forward pass
            with autocast_context():
                outputs = model(**batch)
                loss = outputs.loss

            # Correct accumulation window size,
            # including the last incomplete window
            group_start = (
                batch_idx // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_size = min(
                GRAD_ACCUM_STEPS,
                len(train_dl) - group_start
            )

            scaled_loss = loss / group_size

            # Backward pass
            scaler.scale(scaled_loss).backward()

            batch_size = batch["labels"].size(0)

            epoch_loss_sum += (
                loss.detach() * batch_size
            )
            epoch_samples += batch_size

            # Optimizer update
            should_step = (
                (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
                or (batch_idx + 1) == len(train_dl)
            )

            if should_step:

                # Unscale before gradient clipping
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(
                    set_to_none=True
                )

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        epoch_time = time.perf_counter() - start_time
        total_training_time += epoch_time

        avg_train_loss = (
            epoch_loss_sum.item() / epoch_samples
        )

        # 9. Validation
        val_metrics = evaluate_model(
            model,
            val_dl
        )

        epoch_result = {
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "epoch_time": epoch_time,
            **val_metrics
        }

        history.append(epoch_result)

        print(
            f"\nEpoch {epoch}/{EPOCHS}"
            f"\nTrain Loss: {avg_train_loss:.4f}"
            f"\nVal Loss: {val_metrics['val_loss']:.4f}"
            f"\nVal Accuracy: {val_metrics['val_accuracy']:.4f}"
            f"\nVal Macro-F1: {val_metrics['val_macro_f1']:.4f}"
            f"\nTime: {epoch_time:.2f}s"
        )

        # 10. Save best model
        if val_metrics["val_macro_f1"] > best_f1:
            best_f1 = val_metrics["val_macro_f1"]
            best_epoch = epoch

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "val_macro_f1": best_f1,
                    "model_name": MODEL_NAME
                },
                checkpoint_path
            )

            print("Best checkpoint saved!")

    # 11. GPU memory statistics
    peak_memory_gb = 0.0

    if DEVICE.type == "cuda":
        peak_memory_gb = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

    # 12. Results
    result = {
        "model": model_name,
        "parameters": total_params,
        "trainable_parameters": trainable_params,
        "training_time": total_training_time,
        "peak_vram_gb": peak_memory_gb,
        "best_epoch": best_epoch,
        "best_val_f1": best_f1,
        "checkpoint": str(checkpoint_path),
        "history": history
    }

    print(f"\nCompleted: {model_name}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best Validation F1: {best_f1:.4f}")
    print(f"Training Time: {total_training_time:.2f}s")
    print(f"Peak Reserved VRAM: {peak_memory_gb:.2f} GB")

    # 13. Release memory
    del model, optimizer, scheduler, scaler
    del train_dl, val_dl

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

In [18]:
result_a = train_one_model(
    model_name="transformer_from_scratch",
    model = model_a
)


Training: transformer_from_scratch
Parameters: 66,955,010
Epochs: 3
Optimizer steps: 1860


transformer_from_scratch | Epoch 1/3: 100%|██████████| 4958/4958 [53:00<00:00,  1.56it/s] 



Epoch 1/3
Train Loss: 0.7090
Val Loss: 0.6933
Val Accuracy: 0.5019
Val Macro-F1: 0.3342
Time: 3180.47s
Best checkpoint saved!


transformer_from_scratch | Epoch 2/3: 100%|██████████| 4958/4958 [52:56<00:00,  1.56it/s] 



Epoch 2/3
Train Loss: 0.6935
Val Loss: 0.6931
Val Accuracy: 0.5019
Val Macro-F1: 0.3342
Time: 3176.63s


transformer_from_scratch | Epoch 3/3: 100%|██████████| 4958/4958 [52:57<00:00,  1.56it/s] 



Epoch 3/3
Train Loss: 0.6932
Val Loss: 0.6934
Val Accuracy: 0.5019
Val Macro-F1: 0.3342
Time: 3177.90s

Completed: transformer_from_scratch
Best Epoch: 1
Best Validation F1: 0.3342
Training Time: 9535.00s
Peak Reserved VRAM: 1.37 GB


In [ ]:
result_b = train_one_model(
    model_name="pretrained_transformer",
    model=model_b
)


Training: pretrained_transformer
Parameters: 66,955,010
Epochs: 3
Optimizer steps: 1860


pretrained_transformer | Epoch 1/3:  22%|██▏       | 1111/4958 [11:51<40:35,  1.58it/s] 

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Transformer From Scratch",
        "Parameters": result_a["parameters"],
        "Training Time (s)": result_a["training_time"],
        "Best Epoch": result_a["best_epoch"],
        "Validation F1": result_a["best_val_f1"],
        "Peak VRAM (GB)": result_a["peak_vram_gb"]
    },
    {
        "Model": "Pretrained Transformer",
        "Parameters": result_b["parameters"],
        "Training Time (s)": result_b["training_time"],
        "Best Epoch": result_b["best_epoch"],
        "Validation F1": result_b["best_val_f1"],
        "Peak VRAM (GB)": result_b["peak_vram_gb"]
    }
])

display(comparison_df)

comparison_df.to_csv(
    OUTPUT_DIR / "experiment_1_result.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

history_a = pd.DataFrame(result_a["history"])
history_b = pd.DataFrame(result_b["history"])

plt.figure(figsize=(8, 5))

plt.plot(
    history_a["epoch"],
    history_a["train_loss"],
    marker="o",
    label="From Scratch"
)

plt.plot(
    history_b["epoch"],
    history_b["train_loss"],
    marker="s",
    label="Pretrained"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex1_training_loss.png",
    dpi=150
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_a["epoch"],
    history_a["val_macro_f1"],
    marker="o",
    label="From Scratch"
)

plt.plot(
    history_b["epoch"],
    history_b["val_macro_f1"],
    marker="s",
    label="Pretrained"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")
plt.title("Validation F1 Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex_1validation_f1.png",
    dpi=150
)

plt.show()

# Experiment 2 — Feature Extraction vs Fine-Tuning

In [ ]:
import torch

from transformers import (
    AutoModelForSequenceClassification,
    set_seed
)

NUM_LABELS = 2

ID2LABEL = {
    0: "negative",
    1: "positive"
}

LABEL2ID = {
    "negative": 0,
    "positive": 1
}


## Model preparation 

In [ ]:
def build_frozen_model():
    set_seed(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    # Freeze pretrained encoder
    for param in model.base_model.parameters():
        param.requires_grad = False

    return model

In [ ]:
def build_finetuning_model():
    set_seed(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    # Enable training for all parameters
    for param in model.parameters():
        param.requires_grad = True

    return model

In [ ]:
def count_parameters(model):
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    frozen_params = total_params - trainable_params

    return {
        "total": total_params,
        "trainable": trainable_params,
        "frozen": frozen_params
    }

In [ ]:
model_frozen = build_frozen_model()
model_finetuning = build_finetuning_model()

params_frozen = count_parameters(model_frozen)
params_finetuning = count_parameters(model_finetuning)

print("Mode A — Frozen")
print(params_frozen)

print("\nMode B — Fine-Tuning")
print(params_finetuning)

## Checking freezing condition

In [ ]:
# Same architecture and number of parameters
assert params_frozen["total"] == params_finetuning["total"]

# Frozen encoder must not be trainable
assert all(
    not p.requires_grad
    for p in model_frozen.base_model.parameters()
)

# Classification head must remain trainable
encoder_param_ids = {
    id(p) for p in model_frozen.base_model.parameters()
}

head_params = [
    p for p in model_frozen.parameters()
    if id(p) not in encoder_param_ids
]

assert len(head_params) > 0
assert all(p.requires_grad for p in head_params)

# All parameters are trainable in fine-tuning mode
assert all(
    p.requires_grad
    for p in model_finetuning.parameters()
)

print("Model configuration verification passed!")

## Check weight

In [ ]:
state_frozen = model_frozen.state_dict()
state_finetuning = model_finetuning.state_dict()

assert state_frozen.keys() == state_finetuning.keys()

same_weights = all(
    torch.equal(
        state_frozen[name],
        state_finetuning[name]
    )
    for name in state_frozen
)

print("Identical initial weights:", same_weights)

assert same_weights, "Initial weights are different!"

In [ ]:
batch = next(iter(val_loader))

# Use two samples
test_batch = {
    key: value[:2]
    for key, value in batch.items()
}

def test_forward(model, batch, name):
    model.eval()

    with torch.no_grad():
        outputs = model(**batch)

    print(f"\n{name}")
    print("Logits shape:", outputs.logits.shape)
    print("Loss:", outputs.loss.item())

    assert outputs.logits.shape == (2, NUM_LABELS)


test_forward(
    model_frozen,
    test_batch,
    "Mode A — Frozen"
)

test_forward(
    model_finetuning,
    test_batch,
    "Mode B — Fine-Tuning"
)

In [ ]:
import gc

del model_frozen
del model_finetuning
del state_frozen
del state_finetuning

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleanup completed.")

## Training và Validation

In [ ]:
SEED = 42
EPOCHS = 3

MICRO_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

OUTPUT_DIR = Path("./results/experiment2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print(
    "Effective batch size:",
    MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
)

In [ ]:
def create_train_loader(dataset):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=data_collator,
        pin_memory=(DEVICE.type == "cuda"),
        num_workers=0
    )


def create_val_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
        pin_memory=(DEVICE.type == "cuda"),
        num_workers=0
    )

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    exp_train_dataset = Subset(
        train_dataset,
        range(min(128, len(train_dataset)))
    )

    exp_val_dataset = Subset(
        val_dataset,
        range(min(64, len(val_dataset)))
    )
else:
    exp_train_dataset = train_dataset
    exp_val_dataset = val_dataset

print("Train samples:", len(exp_train_dataset))
print("Validation samples:", len(exp_val_dataset))

def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )

    return nullcontext()


@torch.inference_mode()
def evaluate_model(model, val_loader):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_preds = []
    all_labels = []

    for batch in val_loader:
        batch = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }

        with amp_context():
            outputs = model(**batch)

        labels = batch["labels"]
        preds = outputs.logits.argmax(dim=-1)

        batch_size = labels.size(0)

        total_loss += outputs.loss.item() * batch_size
        total_samples += batch_size

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "val_loss": total_loss / total_samples,

        "val_accuracy": accuracy_score(
            all_labels, all_preds
        ),

        "val_f1": f1_score(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0
        ),

        "val_positive_f1": f1_score(
            all_labels,
            all_preds,
            pos_label=1,
            zero_division=0
        )
    }

In [ ]:
def train_experiment2(model_name, model_builder, mode):
    assert mode in ("frozen", "finetuning")

    # 1. Cleanup and initialization
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    model = model_builder()

    # Gradient checkpointing only for full fine-tuning
    if mode == "finetuning":
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )

    model.to(DEVICE)

    # 2. Verify trainable parameters
    total_params = sum(
        p.numel() for p in model.parameters()
    )

    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    trainable_count = sum(
        p.numel() for p in trainable_params
    )

    if mode == "frozen":
        assert all(
            not p.requires_grad
            for p in model.base_model.parameters()
        )
    else:
        assert all(
            p.requires_grad
            for p in model.parameters()
        )

    # 3. Create DataLoaders
    train_loader = create_train_loader(exp_train_dataset)
    val_loader = create_val_loader(exp_val_dataset)

    # 4. Optimizer
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        foreach=False
    )

    # 5. Scheduler
    steps_per_epoch = math.ceil(
        len(train_loader) / GRAD_ACCUM_STEPS
    )

    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # 6. Mixed precision
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    history = []
    best_f1 = -1.0
    best_epoch = None

    total_train_time = 0.0
    total_val_time = 0.0
    optimizer_steps = 0

    suffix = "_smoke" if SMOKE_TEST else ""
    checkpoint_path = (
        OUTPUT_DIR / f"{model_name}{suffix}_best.pt"
    )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    print(f"\n{'=' * 60}")
    print(f"Model: {model_name}")
    print(f"Mode: {mode}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_count:,}")
    print(f"{'=' * 60}")

    # 7. Training
    for epoch in range(1, EPOCHS + 1):

        model.train()

        # Frozen encoder must remain in eval mode
        if mode == "frozen":
            model.base_model.eval()

        optimizer.zero_grad(set_to_none=True)

        train_loss_sum = torch.zeros(
            (), device=DEVICE
        )
        train_samples = 0

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        progress = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"{model_name} | Epoch {epoch}/{EPOCHS}"
        )

        for batch_idx, batch in progress:

            batch = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
            }

            # Forward
            with amp_context():
                outputs = model(**batch)
                loss = outputs.loss

            batch_size = batch["labels"].size(0)

            # Number of samples in the current
            # gradient accumulation group
            group_start_batch = (
                batch_idx // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_start_sample = (
                group_start_batch * MICRO_BATCH_SIZE
            )

            samples_in_group = min(
                GRAD_ACCUM_STEPS * MICRO_BATCH_SIZE,
                len(exp_train_dataset) - group_start_sample
            )

            # Normalize by actual accumulated samples
            normalized_loss = (
                loss * batch_size / samples_in_group
            )

            # Backward
            scaler.scale(normalized_loss).backward()

            train_loss_sum += (
                loss.detach().float() * batch_size
            )
            train_samples += batch_size

            # Update after gradient accumulation
            should_step = (
                (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
                or batch_idx + 1 == len(train_loader)
            )

            if should_step:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    trainable_params,
                    MAX_GRAD_NORM
                )

                old_scale = scaler.get_scale()

                scaler.step(optimizer)
                scaler.update()

                # Do not advance scheduler if AMP
                # skipped the optimizer step
                step_succeeded = (
                    not USE_AMP
                    or scaler.get_scale() >= old_scale
                )

                if step_succeeded:
                    scheduler.step()
                    optimizer_steps += 1

                optimizer.zero_grad(set_to_none=True)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        epoch_train_time = (
            time.perf_counter() - start_time
        )

        total_train_time += epoch_train_time

        avg_train_loss = (
            train_loss_sum.item() / train_samples
        )

        # 8. Validation
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        val_start = time.perf_counter()

        val_metrics = evaluate_model(
            model,
            val_loader
        )

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        epoch_val_time = (
            time.perf_counter() - val_start
        )
        total_val_time += epoch_val_time

        # 9. Save history
        epoch_result = {
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "train_time": epoch_train_time,
            **val_metrics
        }

        history.append(epoch_result)

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {val_metrics['val_loss']:.4f}")
        print(f"Val Accuracy: {val_metrics['val_accuracy']:.4f}")
        print(f"Val F1: {val_metrics['val_f1']:.4f}")
        print(f"Training Time: {epoch_train_time:.2f}s")

        # 10. Save best checkpoint
        if val_metrics["val_f1"] > best_f1:
            best_f1 = val_metrics["val_f1"]
            best_epoch = epoch

            torch.save(
                {
                    "model_state_dict": {
                        k: v.detach().cpu()
                        for k, v in model.state_dict().items()
                    },
                    "epoch": epoch,
                    "val_f1": best_f1,
                    "model_name": MODEL_NAME,
                    "mode": mode
                },
                checkpoint_path
            )

            print("Best checkpoint saved!")

    # 11. GPU memory
    peak_vram_gb = 0.0

    if DEVICE.type == "cuda":
        peak_vram_gb = (
            torch.cuda.max_memory_reserved() / 1024**3
        )

    # 12. Final results
    result = {
        "mode": mode,
        "model": model_name,
        "total_params": total_params,
        "trainable_params": trainable_count,
        "training_time": total_train_time,
        "validation_time": total_val_time,
        "peak_vram_gb": peak_vram_gb,
        "optimizer_steps": optimizer_steps,
        "best_epoch": best_epoch,
        "best_val_f1": best_f1,
        "checkpoint": str(checkpoint_path),
        "history": history
    }

    print(f"\nCompleted: {model_name}")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best Validation F1: {best_f1:.4f}")
    print(f"Training Time: {total_train_time:.2f}s")
    print(f"Peak VRAM: {peak_vram_gb:.2f} GB")

    # 13. Cleanup
    del model, optimizer, scheduler, scaler
    del train_loader, val_loader

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

In [ ]:
result_frozen = train_experiment2(
    model_name="frozen_transformer",
    model_builder=build_frozen_model,
    mode="frozen"
)
result_finetuning = train_experiment2(
    model_name="finetuned_transformer",
    model_builder=build_finetuning_model,
    mode="finetuning"
)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Mode": "Frozen",
        "Encoder trainable": "No",
        "Trainable Parameters": result_frozen["trainable_params"],
        "Best Validation F1": result_frozen["best_val_f1"],
        "Training Time (s)": result_frozen["training_time"],
        "Peak VRAM (GB)": result_frozen["peak_vram_gb"]
    },
    {
        "Mode": "Fine-Tuning",
        "Encoder trainable": "Yes",
        "Trainable Parameters": result_finetuning["trainable_params"],
        "Best Validation F1": result_finetuning["best_val_f1"],
        "Training Time (s)": result_finetuning["training_time"],
        "Peak VRAM (GB)": result_finetuning["peak_vram_gb"]
    }
])

display(comparison_df)

comparison_df.to_csv(
    OUTPUT_DIR / "ex2_validation_comparison.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

history_frozen = pd.DataFrame(
    result_frozen["history"]
)

history_finetuning = pd.DataFrame(
    result_finetuning["history"]
)

plt.figure(figsize=(8, 5))

plt.plot(
    history_frozen["epoch"],
    history_frozen["train_loss"],
    marker="o",
    label="Frozen"
)

plt.plot(
    history_finetuning["epoch"],
    history_finetuning["train_loss"],
    marker="s",
    label="Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Frozen vs Fine-Tuning — Training Loss")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex2_training_loss.png",
    dpi=150
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_frozen["epoch"],
    history_frozen["val_f1"],
    marker="o",
    label="Frozen"
)

plt.plot(
    history_finetuning["epoch"],
    history_finetuning["val_f1"],
    marker="s",
    label="Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")
plt.title("Frozen vs Fine-Tuning — Validation F1")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex_2validation_f1.png",
    dpi=150
)

plt.show()

## Test

In [ ]:
import gc
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from contextlib import nullcontext
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = Path("./results/experiment2")
MICRO_BATCH_SIZE = 2

USE_AMP = DEVICE.type == "cuda"

# Check checkpoints
FROZEN_CHECKPOINT = (
    OUTPUT_DIR / "frozen_transformer_best.pt"
)

FINETUNING_CHECKPOINT = (
    OUTPUT_DIR / "finetuned_transformer_best.pt"
)

assert FROZEN_CHECKPOINT.exists(), (
    f"Missing checkpoint: {FROZEN_CHECKPOINT}"
)

assert FINETUNING_CHECKPOINT.exists(), (
    f"Missing checkpoint: {FINETUNING_CHECKPOINT}"
)

print("Device:", DEVICE)
print("Both checkpoints found!")

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
@torch.inference_mode()
def evaluate_on_test(model, test_loader):
    model.eval()

    all_predictions = []
    all_labels = []

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in test_loader:
        batch = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }

        if USE_AMP:
            amp_context = torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            )
        else:
            amp_context = nullcontext()

        with amp_context:
            outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=-1
        )

        labels = batch["labels"]

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.cpu().tolist()
        )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    inference_time = time.perf_counter() - start_time

    # Calculate metrics
    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    positive_f1 = f1_score(
        all_labels,
        all_predictions,
        pos_label=1,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_f1": positive_f1,
        "inference_time": inference_time,
        "predictions": all_predictions,
        "labels": all_labels
    }

In [ ]:
def load_and_evaluate(model_builder, checkpoint_path, mode):
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print(f"\nEvaluating: {mode}")

    # 1. Load checkpoint into CPU
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    # 2. Initialize model architecture
    model = model_builder()

    # 3. Restore best weights
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    best_epoch = checkpoint["epoch"]
    best_val_f1 = checkpoint["val_f1"]

    del checkpoint

    # 4. Count parameters
    total_params = sum(
        p.numel() for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    # 5. Move model to GPU
    model = model.to(DEVICE)

    # 6. Evaluate
    metrics = evaluate_on_test(
        model,
        test_loader
    )

    metrics.update({
        "mode": mode,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "total_params": total_params,
        "trainable_params": trainable_params
    })

    print(f"Best Epoch: {best_epoch}")
    print(f"Validation F1: {best_val_f1:.4f}")
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")
    print(f"Test Precision: {metrics['precision']:.4f}")
    print(f"Test Recall: {metrics['recall']:.4f}")
    print(f"Test F1: {metrics['f1']:.4f}")
    print(f"Inference Time: {metrics['inference_time']:.2f}s")

    # 7. Release memory
    del model
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return metrics

In [ ]:
test_frozen = load_and_evaluate(
    model_builder=build_frozen_model,
    checkpoint_path=FROZEN_CHECKPOINT,
    mode="Frozen"
)
test_finetuning = load_and_evaluate(
    model_builder=build_finetuning_model,
    checkpoint_path=FINETUNING_CHECKPOINT,
    mode="Fine-Tuning"
)

In [ ]:
# Training time from Step 3
if "result_frozen" in globals() and "result_finetuning" in globals():
    frozen_train_time = result_frozen["training_time"]
    finetuning_train_time = result_finetuning["training_time"]

else:
    # Fallback: load saved validation comparison
    saved_df = pd.read_csv(
        OUTPUT_DIR / "ex2_validation_comparison.csv"
    )

    frozen_train_time = saved_df.loc[
        saved_df["Mode"] == "Frozen",
        "Training Time (s)"
    ].iloc[0]

    finetuning_train_time = saved_df.loc[
        saved_df["Mode"] == "Fine-Tuning",
        "Training Time (s)"
    ].iloc[0]


results_df = pd.DataFrame([
    {
        "Mode": "Frozen",
        "Encoder trainable?": "No",
        "Accuracy": test_frozen["accuracy"],
        "Precision": test_frozen["precision"],
        "Recall": test_frozen["recall"],
        "F1": test_frozen["f1"],
        "Training time (s)": frozen_train_time,
        "Total Parameters": test_frozen["total_params"],
        "Trainable Parameters": test_frozen["trainable_params"]
    },
    {
        "Mode": "Fine-Tuning",
        "Encoder trainable?": "Yes",
        "Accuracy": test_finetuning["accuracy"],
        "Precision": test_finetuning["precision"],
        "Recall": test_finetuning["recall"],
        "F1": test_finetuning["f1"],
        "Training time (s)": finetuning_train_time,
        "Total Parameters": test_finetuning["total_params"],
        "Trainable Parameters": test_finetuning["trainable_params"]
    }
])

display(results_df.round(4))

results_df.to_csv(
    OUTPUT_DIR / "ex2_test_results.csv",
    index=False
)

In [ ]:
for name, result in [
    ("Frozen", test_frozen),
    ("Fine-Tuning", test_finetuning)
]:
    print(f"\n{'=' * 50}")
    print(f"{name} — Classification Report")
    print(f"{'=' * 50}")

    report = classification_report(
        result["labels"],
        result["predictions"],
        target_names=["negative", "positive"],
        zero_division=0
    )

    print(report)

In [ ]:
for name, result in [
    ("Frozen", test_frozen),
    ("Fine-Tuning", test_finetuning)
]:
    cm = confusion_matrix(
        result["labels"],
        result["predictions"],
        labels=[0, 1]
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["negative", "positive"]
    )

    disp.plot(
        cmap="Blues",
        values_format="d"
    )

    plt.title(f"{name} — Confusion Matrix")
    plt.tight_layout()

    filename = name.lower().replace("-", "_")

    plt.savefig(
        OUTPUT_DIR / f"{filename}_confusion_matrix.png",
        dpi=150
    )

    plt.show()

In [ ]:
accuracy_diff = (
    test_finetuning["accuracy"]
    - test_frozen["accuracy"]
)

f1_diff = (
    test_finetuning["f1"]
    - test_frozen["f1"]
)

time_ratio = (
    finetuning_train_time / frozen_train_time
)

print("Fine-Tuning vs Frozen")
print(f"Accuracy difference: {accuracy_diff:+.4f}")
print(f"F1 difference: {f1_diff:+.4f}")
print(f"Training time ratio: {time_ratio:.2f}x")

if f1_diff > 0:
    print("Fine-Tuning achieved higher test F1.")
elif f1_diff < 0:
    print("Frozen achieved higher test F1.")
else:
    print("Both modes achieved equal test F1.")

# Experiment 3: Experiment 3 — Amount of Training Data

## Data preparation

In [ ]:
import numpy as np

from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

SEED = 42

# Reuse training dataset from Experiment 1
labels = np.asarray(
    train_dataset["label"],
    dtype=np.int64
)

all_indices = np.arange(len(train_dataset))

# 50% subset
idx_50, _ = train_test_split(
    all_indices,
    train_size=0.5,
    random_state=SEED,
    stratify=labels
)

# 25% subset (half of 50%)
idx_25, _ = train_test_split(
    idx_50,
    train_size=0.5,
    random_state=SEED,
    stratify=labels[idx_50]
)

# 10% subset (40% of 25%)
idx_10, _ = train_test_split(
    idx_25,
    train_size=0.4,
    random_state=SEED,
    stratify=labels[idx_25]
)

training_subsets = {
    10: Subset(train_dataset, idx_10.tolist()),
    25: Subset(train_dataset, idx_25.tolist()),
    50: Subset(train_dataset, idx_50.tolist()),
    100: train_dataset
}

for percentage, subset in training_subsets.items():
    print(
        f"{percentage}%: {len(subset):,} samples"
    )

assert set(idx_10).issubset(set(idx_25))
assert set(idx_25).issubset(set(idx_50))
assert set(idx_50).issubset(set(all_indices))

print("Nested subsets verified!")

for percentage, subset in training_subsets.items():
    subset_labels = [
        int(subset[i]["label"])
        for i in range(len(subset))
    ]

    unique, counts = np.unique(
        subset_labels,
        return_counts=True
    )

    print(f"\n{percentage}%")
    print(dict(zip(unique, counts)))

## Model Initialization

In [ ]:

model_builder = build_finetuning_model

print("Model:", MODEL_NAME)
# print("Model builder:", model_builder.__name__)

## Training and Validation

In [ ]:
from pathlib import Path
import pandas as pd

# Experiment 3 configuration
EPOCHS = 3

MICRO_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# Output directory
OUTPUT_DIR = Path("./results/experiment3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use full training data, not smoke test
SMOKE_TEST = False

# Fixed validation set for all runs
exp_val_dataset = val_dataset

print("Training configuration:")
print("Epochs:", EPOCHS)
print("Micro-batch size:", MICRO_BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
experiment3_results = {}

for percentage in [10, 25, 50, 100]:

    # Select training subset
    exp_train_dataset = training_subsets[percentage]

    print(f"\n{'=' * 60}")
    print(f"Training with {percentage}% data")
    print(f"Training samples: {len(exp_train_dataset):,}")
    print(f"Validation samples: {len(exp_val_dataset):,}")
    print(f"{'=' * 60}")

    # Train a new pretrained model
    result = train_experiment2(
        model_name=f"pretrained_{percentage}pct",
        model_builder=model_builder,
        mode="finetuning"
    )

    # Store training results
    experiment3_results[percentage] = result

print("\nAll training runs completed!")

In [ ]:
training_results = []

for percentage in [10, 25, 50, 100]:
    result = experiment3_results[percentage]

    training_results.append({
        "Training Data": f"{percentage}%",
        "Samples": len(training_subsets[percentage]),
        "Best Epoch": result["best_epoch"],
        "Validation F1": result["best_val_f1"],
        "Training Time (s)": result["training_time"],
        "Peak VRAM (GB)": result["peak_vram_gb"]
    })

training_df = pd.DataFrame(training_results)

display(training_df.round(4))

training_df.to_csv(
    OUTPUT_DIR / "ex3_training_results.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

for percentage in [10, 25, 50, 100]:

    history = pd.DataFrame(
        experiment3_results[percentage]["history"]
    )

    plt.plot(
        history["epoch"],
        history["val_f1"],
        marker="o",
        label=f"{percentage}% Data"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")
plt.title("Validation F1 Across Training Data Sizes")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex3_validation_f1_by_epoch.png",
    dpi=150
)

plt.show()

## Test

In [ ]:
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader

# Same test set for all training subsets
test_loader = DataLoader(
    test_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
# Store test results
experiment3_test_results = {}

for percentage in [10, 25, 50, 100]:

    print(f"\n{'=' * 60}")
    print(f"Evaluating pretrained model — {percentage}% data")
    print(f"{'=' * 60}")

    # Get best checkpoint from Step 3
    train_result = experiment3_results[percentage]

    checkpoint_path = Path(
        train_result["checkpoint"]
    )

    assert checkpoint_path.exists(), (
        f"Checkpoint not found: {checkpoint_path}"
    )

    # Load checkpoint and evaluate on test set
    test_result = load_and_evaluate(
        model_builder=model_builder,
        checkpoint_path=checkpoint_path,
        mode=f"Pretrained {percentage}%"
    )

    # Store results
    experiment3_test_results[percentage] = test_result

print("\nAll test evaluations completed!")

In [ ]:
rows = []

for percentage in [10, 25, 50, 100]:

    test_result = experiment3_test_results[percentage]
    train_result = experiment3_results[percentage]

    rows.append({
        "Training Data (%)": percentage,
        "Samples": len(training_subsets[percentage]),
        "Accuracy": test_result["accuracy"],
        "Precision": test_result["precision"],
        "Recall": test_result["recall"],
        "F1": test_result["f1"],
        "Training Time (s)": train_result["training_time"],
        "Best Epoch": train_result["best_epoch"]
    })

results_df = pd.DataFrame(rows)

display(results_df.round(4))

# Save results
OUTPUT_DIR = Path("./results/experiment3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "ex3_test_results.csv",
    index=False
)

In [ ]:
from sklearn.metrics import classification_report

for percentage in [10, 25, 50, 100]:

    result = experiment3_test_results[percentage]

    print(f"\n{'=' * 60}")
    print(f"Training Data: {percentage}%")
    print(f"{'=' * 60}")

    print(
        classification_report(
            result["labels"],
            result["predictions"],
            labels=[0, 1],
            target_names=["negative", "positive"],
            zero_division=0
        )
    )

In [ ]:
analysis_df = results_df[
    ["Training Data (%)", "Accuracy", "F1"]
].copy()

analysis_df["Accuracy Change"] = (
    analysis_df["Accuracy"].diff()
)

analysis_df["F1 Change"] = (
    analysis_df["F1"].diff()
)

display(analysis_df.round(4))

## Data-Efficiency Curve

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("./results/experiment3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse results_df from Step 4
# Or load from CSV if notebook was restarted
results_df = pd.read_csv(OUTPUT_DIR / "test_results.csv")

results_df = results_df.sort_values(
    "Training Data (%)"
).reset_index(drop=True)

percentages = results_df["Training Data (%)"].to_numpy()
accuracies = results_df["Accuracy"].to_numpy()
f1_scores = results_df["F1"].to_numpy()

display(
    results_df[
        ["Training Data (%)", "Samples", "Accuracy", "F1"]
    ].round(4)
)

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    percentages,
    accuracies,
    marker="o",
    linewidth=2,
    markersize=8,
    label="Accuracy"
)

plt.plot(
    percentages,
    f1_scores,
    marker="s",
    linewidth=2,
    markersize=8,
    label="Macro-F1"
)

plt.xlabel("Training Data (%)", fontsize=12)
plt.ylabel("Performance", fontsize=12)

plt.title(
    "Pretrained Transformer — Data Efficiency",
    fontsize=14
)

plt.xticks(percentages)
plt.ylim(0, 1)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "data_efficiency_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
analysis_df = results_df[
    ["Training Data (%)", "Accuracy", "F1"]
].copy()

analysis_df["Accuracy Change"] = (
    analysis_df["Accuracy"].diff()
)

analysis_df["F1 Change"] = (
    analysis_df["F1"].diff()
)

display(analysis_df.round(4))

f1_10 = results_df.loc[
    results_df["Training Data (%)"] == 10,
    "F1"
].iloc[0]

f1_100 = results_df.loc[
    results_df["Training Data (%)"] == 100,
    "F1"
].iloc[0]

accuracy_10 = results_df.loc[
    results_df["Training Data (%)"] == 10,
    "Accuracy"
].iloc[0]

accuracy_100 = results_df.loc[
    results_df["Training Data (%)"] == 100,
    "Accuracy"
].iloc[0]

print(
    f"Accuracy improvement (10% → 100%): "
    f"{accuracy_100 - accuracy_10:+.4f}"
)

print(
    f"F1 improvement (10% → 100%): "
    f"{f1_100 - f1_10:+.4f}"
)

In [ ]:
f1_10 = results_df.loc[
    results_df["Training Data (%)"] == 10,
    "F1"
].iloc[0]

f1_100 = results_df.loc[
    results_df["Training Data (%)"] == 100,
    "F1"
].iloc[0]

accuracy_10 = results_df.loc[
    results_df["Training Data (%)"] == 10,
    "Accuracy"
].iloc[0]

accuracy_100 = results_df.loc[
    results_df["Training Data (%)"] == 100,
    "Accuracy"
].iloc[0]

print(
    f"Accuracy improvement (10% → 100%): "
    f"{accuracy_100 - accuracy_10:+.4f}"
)

print(
    f"F1 improvement (10% → 100%): "
    f"{f1_100 - f1_10:+.4f}"
)

In [ ]:
relative_df = results_df[
    ["Training Data (%)", "Accuracy", "F1"]
].copy()

f1_full = relative_df.loc[
    relative_df["Training Data (%)"] == 100,
    "F1"
].iloc[0]

relative_df["Relative F1 (%)"] = (
    relative_df["F1"] / f1_full * 100
    if f1_full > 0
    else np.nan
)

display(relative_df.round(2))

In [ ]:
f1_values = results_df.set_index("Training Data (%)")["F1"]

f1_10 = f1_values.loc[10]
f1_25 = f1_values.loc[25]
f1_50 = f1_values.loc[50]
f1_100 = f1_values.loc[100]

print("DATA-EFFICIENCY ANALYSIS")
print("=" * 50)

print(f"F1 at 10%: {f1_10:.4f}")
print(f"F1 at 25%: {f1_25:.4f}")
print(f"F1 at 50%: {f1_50:.4f}")
print(f"F1 at 100%: {f1_100:.4f}")

print("\nPerformance improvements:")
print(f"10% → 25%: {f1_25 - f1_10:+.4f}")
print(f"25% → 50%: {f1_50 - f1_25:+.4f}")
print(f"50% → 100%: {f1_100 - f1_50:+.4f}")

if f1_100 > f1_10:
    print(
        "\nOverall F1 increased as training data "
        "grew from 10% to 100%."
    )
elif f1_100 < f1_10:
    print(
        "\nOverall F1 decreased from 10% to 100%. "
        "Investigate optimization and overfitting."
    )
else:
    print(
        "\nF1 at 10% and 100% is equal."
    )

monotonic = np.all(
    np.diff(f1_values.loc[[10, 25, 50, 100]].to_numpy()) >= 0
)

print("Monotonic non-decreasing F1:", monotonic)

# Experiment 4: Maximum Sequence Length

## Data preparation

In [ ]:
import pandas as pd
import numpy as np

SEED = 42
MAX_LENGTHS = [64, 128, 256]

# Reuse cleaned datasets from Experiment 1
raw_train_df = train_df.copy().reset_index(drop=True)
raw_val_df = val_df.copy().reset_index(drop=True)
raw_test_df = test_df.copy().reset_index(drop=True)

# Verify required columns
for name, df_split in [
    ("Train", raw_train_df),
    ("Validation", raw_val_df),
    ("Test", raw_test_df)
]:
    assert {"review", "label"}.issubset(df_split.columns)

    assert df_split["review"].notna().all()
    assert df_split["label"].notna().all()

    assert df_split["label"].isin([0, 1]).all()

    print(f"{name}: {len(df_split):,} samples")

In [ ]:
train_texts = set(raw_train_df["review"])
val_texts = set(raw_val_df["review"])
test_texts = set(raw_test_df["review"])

train_val_overlap = len(train_texts & val_texts)
train_test_overlap = len(train_texts & test_texts)
val_test_overlap = len(val_texts & test_texts)

print("Train-Val overlap:", train_val_overlap)
print("Train-Test overlap:", train_test_overlap)
print("Val-Test overlap:", val_test_overlap)

assert train_val_overlap == 0
assert train_test_overlap == 0
assert val_test_overlap == 0

print("\nNo data leakage between splits!")

for name, df_split in [
    ("Train", raw_train_df),
    ("Validation", raw_val_df),
    ("Test", raw_test_df)
]:
    print(f"\n{name}")
    print("Total:", len(df_split))
    print(df_split["label"].value_counts().sort_index())

In [ ]:
from datasets import Dataset, DatasetDict

raw_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        raw_train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        raw_val_df,
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        raw_test_df,
        preserve_index=False
    )
})

print(raw_dataset)

In [ ]:
sample = raw_dataset["train"][0]

print("Review:", sample["review"][:300])
print("Label:", sample["label"])

In [ ]:
assert len(raw_dataset["train"]) == len(raw_train_df)
assert len(raw_dataset["validation"]) == len(raw_val_df)
assert len(raw_dataset["test"]) == len(raw_test_df)

assert "review" in raw_dataset["train"].column_names
assert "label" in raw_dataset["train"].column_names

print("Step 1 — Dataset Preparation Completed!")

## Tokenization

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MAX_LENGTHS = [64, 128, 256]

# Reuse MODEL_NAME from previous experiments
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)


def prepare_tokenized_dataset(max_length):
    assert max_length in MAX_LENGTHS

    def tokenize_function(batch):
        return tokenizer(
            batch["review"],
            truncation=True,
            max_length=max_length,
            padding=False
        )

    tokenized = raw_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["review"],
        desc=f"Tokenizing with max_length={max_length}"
    )

    tokenized.set_format(
        type="torch",
        columns=[
            "input_ids",
            "attention_mask",
            "label"
        ]
    )

    return tokenized

In [ ]:
tokenized_64 = prepare_tokenized_dataset(
    max_length=64
)

print(tokenized_64)

sample = tokenized_64["train"][0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["label"])

assert len(sample["input_ids"]) <= 64

print("Tokenization test passed!")

In [ ]:
tokenized_datasets = {}

for max_length in MAX_LENGTHS:
    print(f"\nPreparing max_length={max_length}")

    tokenized_datasets[max_length] = (
        prepare_tokenized_dataset(max_length)
    )

print("\nAll tokenized datasets prepared!")

In [ ]:
for max_length, dataset in tokenized_datasets.items():

    print(f"\nChecking max_length={max_length}")

    for split in ["train", "validation", "test"]:

        max_observed_length = max(
            len(sample["input_ids"])
            for sample in dataset[split]
        )

        assert max_observed_length <= max_length

        print(
            f"{split}: "
            f"Maximum observed length = "
            f"{max_observed_length}"
        )

print("\nAll sequence lengths verified!")

In [ ]:
import numpy as np

reference = tokenized_datasets[64]

for max_length in [128, 256]:

    current = tokenized_datasets[max_length]

    for split in ["train", "validation", "test"]:

        # Verify number of samples
        assert len(reference[split]) == len(current[split])

        # Verify identical labels
        assert np.array_equal(
            np.asarray(reference[split]["label"]),
            np.asarray(current[split]["label"])
        ), f"Label mismatch: T=64 vs T={max_length}, split={split}"

print("Dataset consistency verified!")

In [ ]:
import numpy as np
import pandas as pd

def get_original_token_lengths(dataset, batch_size=128):
    lengths = []

    reviews = dataset["review"]

    for start in range(0, len(reviews), batch_size):
        batch = reviews[start:start + batch_size]

        encoded = tokenizer(
            batch,
            truncation=False,
            padding=False,
            return_length=True
        )

        lengths.extend(encoded["length"])

    return np.asarray(lengths)


original_lengths = get_original_token_lengths(
    raw_dataset["train"]
)

print("Average length:", original_lengths.mean())
print("Median length:", np.median(original_lengths))
print("Max length:", original_lengths.max())

## Training and validation

In [ ]:
import gc
import time
import pandas as pd
import torch

from pathlib import Path

# Experiment 4 configuration
MAX_LENGTHS = [64, 128, 256]

SEED = 42
EPOCHS = 3

MICRO_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

SMOKE_TEST = False

# Output directory
OUTPUT_DIR = Path("./results/experiment4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment 4 Configuration")
print("Model:", MODEL_NAME)
print("Max Lengths:", MAX_LENGTHS)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print(
    "Effective Batch Size:",
    MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
)

In [ ]:
experiment4_results = []
experiment4_histories = {}

for max_length in MAX_LENGTHS:

    print(f"\n{'=' * 60}")
    print(f"EXPERIMENT 4 — MAX LENGTH = {max_length}")
    print(f"{'=' * 60}")

    # 1. Select tokenized dataset
    dataset = tokenized_datasets[max_length]

    # 2. Set training and validation datasets
    exp_train_dataset = dataset["train"]
    exp_val_dataset = dataset["validation"]

    print(f"Training samples: {len(exp_train_dataset):,}")
    print(f"Validation samples: {len(exp_val_dataset):,}")

    # 3. Cleanup GPU memory
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 4. Train a fresh pretrained model
    result = train_experiment2(
        model_name=f"pretrained_len_{max_length}",
        model_builder=build_finetuning_model,
        mode="finetuning"
    )

    # 5. Store results
    experiment4_results.append({
        "Max Length": max_length,
        "Training Time (s)": result["training_time"],
        "Validation Time (s)": result["validation_time"],
        "Peak GPU Memory (GB)": result["peak_vram_gb"],
        "Best Epoch": result["best_epoch"],
        "Best Validation F1": result["best_val_f1"],
        "Parameters": result["total_params"],
        "Checkpoint": result["checkpoint"]
    })

    # 6. Store training history
    experiment4_histories[max_length] = result["history"]

    # 7. Save intermediate results
    pd.DataFrame(experiment4_results).to_csv(
        OUTPUT_DIR / "training_results.csv",
        index=False
    )

    pd.DataFrame(result["history"]).to_csv(
        OUTPUT_DIR / f"history_len_{max_length}.csv",
        index=False
    )

    print(f"\nCompleted max_length={max_length}")
    print(f"Best Validation F1: {result['best_val_f1']:.4f}")
    print(f"Training Time: {result['training_time']:.2f}s")
    print(f"Peak GPU Memory: {result['peak_vram_gb']:.2f} GB")

print("\nAll training runs completed!")

In [ ]:
training_df = pd.DataFrame(experiment4_results)

display(
    training_df[
        [
            "Max Length",
            "Best Epoch",
            "Best Validation F1",
            "Training Time (s)",
            "Peak GPU Memory (GB)"
        ]
    ].round(4)
)

training_df.to_csv(
    OUTPUT_DIR / "ex4_training_results.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

for max_length in MAX_LENGTHS:

    history = pd.DataFrame(
        experiment4_histories[max_length]
    )

    plt.plot(
        history["epoch"],
        history["val_f1"],
        marker="o",
        linewidth=2,
        label=f"Max Length = {max_length}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")

plt.title(
    "Experiment 4 — Validation F1 by Epoch"
)

plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "validation_f1_by_epoch.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for max_length in MAX_LENGTHS:

    history = pd.DataFrame(
        experiment4_histories[max_length]
    )

    plt.plot(
        history["epoch"],
        history["train_loss"],
        marker="o",
        linewidth=2,
        label=f"Max Length = {max_length}"
    )

plt.xlabel("Epoch")
plt.ylabel("Training Loss")

plt.title(
    "Experiment 4 — Training Loss by Epoch"
)

plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "training_loss_by_epoch.png",
    dpi=300
)

plt.show()

## Test

In [ ]:
import gc
import torch
import pandas as pd

from pathlib import Path
from torch.utils.data import DataLoader

MAX_LENGTHS = [64, 128, 256]

OUTPUT_DIR = Path("./results/experiment4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load training results from Step 3
training_df = pd.read_csv(
    OUTPUT_DIR / "training_results.csv"
)

print("Device:", DEVICE)
print("Training results:")
display(training_df)

In [ ]:
experiment4_test_results = {}

for max_length in MAX_LENGTHS:

    print(f"\n{'=' * 60}")
    print(f"TEST EVALUATION — MAX LENGTH = {max_length}")
    print(f"{'=' * 60}")

    # 1. Select corresponding tokenized test dataset
    test_dataset_t = tokenized_datasets[max_length]["test"]

    # 2. Create test DataLoader
    test_loader = DataLoader(
        test_dataset_t,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda")
    )

    print(f"Test samples: {len(test_dataset_t):,}")
    print(f"Test batches: {len(test_loader):,}")

    # 3. Get checkpoint from training results
    train_row = training_df.loc[
        training_df["Max Length"] == max_length
    ]

    assert len(train_row) == 1, (
        f"Missing or duplicate training result for T={max_length}"
    )

    checkpoint_path = Path(
        train_row.iloc[0]["Checkpoint"]
    )

    assert checkpoint_path.exists(), (
        f"Checkpoint not found: {checkpoint_path}"
    )

    # 4. Evaluate pretrained Transformer
    test_result = load_and_evaluate(
        model_builder=build_finetuning_model,
        checkpoint_path=checkpoint_path,
        mode=f"Pretrained T={max_length}"
    )

    # 5. Store results
    experiment4_test_results[max_length] = test_result

    print(f"\nCompleted T={max_length}")
    print(f"Accuracy: {test_result['accuracy']:.4f}")
    print(f"Macro-F1: {test_result['f1']:.4f}")
    print(f"Inference time: {test_result['inference_time']:.2f}s")

    # 6. Release memory
    del test_loader
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\nAll test evaluations completed!")

In [ ]:
results = []

for max_length in MAX_LENGTHS:

    # Test metrics
    test_result = experiment4_test_results[max_length]

    # Training metrics
    train_row = training_df.loc[
        training_df["Max Length"] == max_length
    ].iloc[0]

    results.append({
        "Max Length": max_length,
        "Accuracy": test_result["accuracy"],
        "Precision": test_result["precision"],
        "Recall": test_result["recall"],
        "F1": test_result["f1"],
        "Training Time (s)": train_row["Training Time (s)"],
        "GPU Memory (GB)": train_row["Peak GPU Memory (GB)"],
        "Best Epoch": train_row["Best Epoch"],
        "Validation F1": train_row["Best Validation F1"]
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

# Save results
results_df.to_csv(
    OUTPUT_DIR / "ex4_test_results.csv",
    index=False
)

In [ ]:
from sklearn.metrics import classification_report

for max_length in MAX_LENGTHS:

    result = experiment4_test_results[max_length]

    print(f"\n{'=' * 60}")
    print(f"Classification Report — T={max_length}")
    print(f"{'=' * 60}")

    report = classification_report(
        result["labels"],
        result["predictions"],
        labels=[0, 1],
        target_names=["negative", "positive"],
        zero_division=0
    )

    print(report)

In [ ]:
analysis_df = results_df[
    [
        "Max Length",
        "Accuracy",
        "F1",
        "Training Time (s)",
        "GPU Memory (GB)"
    ]
].copy()

analysis_df["Accuracy Change"] = (
    analysis_df["Accuracy"].diff()
)

analysis_df["F1 Change"] = (
    analysis_df["F1"].diff()
)

analysis_df["Training Time Ratio"] = (
    analysis_df["Training Time (s)"]
    / analysis_df["Training Time (s)"].iloc[0]
)

analysis_df["GPU Memory Ratio"] = (
    analysis_df["GPU Memory (GB)"]
    / analysis_df["GPU Memory (GB)"].iloc[0]
    if analysis_df["GPU Memory (GB)"].iloc[0] > 0
    else float("nan")
)

display(analysis_df.round(4))

## Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

OUTPUT_DIR = Path("./results/experiment4")

# Load results from Step 4
results_df = pd.read_csv(
    OUTPUT_DIR / "test_results.csv"
)

results_df = results_df.sort_values(
    "Max Length"
).reset_index(drop=True)

max_lengths = results_df["Max Length"].to_numpy()
accuracies = results_df["Accuracy"].to_numpy()
f1_scores = results_df["F1"].to_numpy()

training_times = results_df[
    "Training Time (s)"
].to_numpy()

gpu_memories = results_df[
    "GPU Memory (GB)"
].to_numpy()

display(results_df.round(4))

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    max_lengths,
    accuracies,
    marker="o",
    linewidth=2,
    markersize=8,
    label="Accuracy"
)

plt.plot(
    max_lengths,
    f1_scores,
    marker="s",
    linewidth=2,
    markersize=8,
    label="Macro-F1"
)

plt.xlabel("Maximum Sequence Length")
plt.ylabel("Performance")

plt.title(
    "Experiment 4 — Performance vs Sequence Length"
)

plt.xticks(max_lengths)
plt.ylim(0, 1)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex4_performance_vs_length.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

bars = plt.bar(
    max_lengths.astype(str),
    training_times
)

plt.xlabel("Maximum Sequence Length")
plt.ylabel("Training Time (seconds)")

plt.title(
    "Experiment 4 — Training Time vs Sequence Length"
)

plt.bar_label(
    bars,
    fmt="%.1f s",
    padding=3
)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex4_training_time_vs_length.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

bars = plt.bar(
    max_lengths.astype(str),
    gpu_memories
)

plt.xlabel("Maximum Sequence Length")
plt.ylabel("Peak GPU Memory (GB)")

plt.title(
    "Experiment 4 — GPU Memory vs Sequence Length"
)

plt.bar_label(
    bars,
    fmt="%.2f GB",
    padding=3
)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ex4_gpu_memory_vs_length.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
analysis_df = results_df[
    [
        "Max Length",
        "Accuracy",
        "F1",
        "Training Time (s)",
        "GPU Memory (GB)"
    ]
].copy()

# Absolute performance changes
analysis_df["Delta Accuracy"] = (
    analysis_df["Accuracy"].diff()
)

analysis_df["Delta F1"] = (
    analysis_df["F1"].diff()
)

# Relative computational cost vs T=64
baseline_time = analysis_df[
    "Training Time (s)"
].iloc[0]

baseline_memory = analysis_df[
    "GPU Memory (GB)"
].iloc[0]

analysis_df["Time Ratio"] = (
    analysis_df["Training Time (s)"]
    / baseline_time
    if baseline_time > 0
    else np.nan
)

analysis_df["Memory Ratio"] = (
    analysis_df["GPU Memory (GB)"]
    / baseline_memory
    if baseline_memory > 0
    else np.nan
)

display(analysis_df.round(4))

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    training_times,
    f1_scores,
    s=100
)

for i, max_length in enumerate(max_lengths):
    plt.annotate(
        f"T={max_length}",
        (
            training_times[i],
            f1_scores[i]
        ),
        xytext=(8, 8),
        textcoords="offset points"
    )

plt.xlabel("Training Time (seconds)")
plt.ylabel("Test Macro-F1")

plt.title(
    "Experiment 4 — Performance vs Training Cost"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "f1_vs_training_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
best_f1_row = results_df.loc[
    results_df["F1"].idxmax()
]

fastest_row = results_df.loc[
    results_df["Training Time (s)"].idxmin()
]

lowest_memory_row = results_df.loc[
    results_df["GPU Memory (GB)"].idxmin()
]

print("=" * 60)
print("EXPERIMENT 4 — ANALYSIS SUMMARY")
print("=" * 60)

print(
    f"Best F1: T={int(best_f1_row['Max Length'])}, "
    f"F1={best_f1_row['F1']:.4f}"
)

print(
    f"Fastest Training: T={int(fastest_row['Max Length'])}, "
    f"Time={fastest_row['Training Time (s)']:.2f}s"
)

if baseline_memory > 0:
    print(
        f"Lowest GPU Memory: "
        f"T={int(lowest_memory_row['Max Length'])}, "
        f"Memory={lowest_memory_row['GPU Memory (GB)']:.2f} GB"
    )

print("\nPerformance Changes:")

for i in range(1, len(results_df)):
    prev = results_df.iloc[i - 1]
    curr = results_df.iloc[i]

    print(
        f"T={int(prev['Max Length'])} → "
        f"T={int(curr['Max Length'])}: "
        f"ΔF1={curr['F1'] - prev['F1']:+.4f}, "
        f"ΔAccuracy={curr['Accuracy'] - prev['Accuracy']:+.4f}"
    )

In [ ]:
analysis_df.to_csv(
    OUTPUT_DIR / "ex4_analysis_results.csv",
    index=False
)

print("Analysis saved successfully!")

# Experiment 5 — Domain Shift

## Data preparation

In [ ]:
# Domain A — IMDb Movie Reviews
imdb_test_df = test_df[["review", "label"]].copy()

imdb_test_df = imdb_test_df.rename(
    columns={"review": "text"}
)

imdb_test_df = imdb_test_df.reset_index(drop=True)

print("IMDb test samples:", len(imdb_test_df))
print(imdb_test_df["label"].value_counts())

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
OOD_TEST_SIZE = 5000

# Load only Yelp test split
yelp_test = load_dataset(
    "fancyzhx/yelp_polarity",
    split="test"
)

print(yelp_test)
print(yelp_test.features)
print(yelp_test[0])
labels = np.asarray(yelp_test["label"])

selected_indices, _ = train_test_split(
    np.arange(len(yelp_test)),
    train_size=OOD_TEST_SIZE,
    random_state=SEED,
    stratify=labels
)

yelp_test = yelp_test.select(
    selected_indices.tolist()
)

print("Yelp test samples:", len(yelp_test))
print(
    np.unique(
        yelp_test["label"],
        return_counts=True
    )
)

In [ ]:
from datasets import Dataset

imdb_test_raw = Dataset.from_pandas(
    imdb_test_df,
    preserve_index=False
)

yelp_test_raw = yelp_test

print("Domain A:", imdb_test_raw)
print("Domain B:", yelp_test_raw)

## Tokenizer

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding
)
from torch.utils.data import DataLoader

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

In [ ]:
imdb_test_tokenized = imdb_test_raw.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing IMDb"
)

yelp_test_tokenized = yelp_test_raw.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing Yelp"
)

print("IMDb:", imdb_test_tokenized)
print("Yelp:", yelp_test_tokenized)

In [ ]:
for dataset in [imdb_test_tokenized, yelp_test_tokenized]:
    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

In [ ]:
imdb_test_loader = DataLoader(
    imdb_test_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

yelp_test_loader = DataLoader(
    yelp_test_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("IMDb batches:", len(imdb_test_loader))
print("Yelp batches:", len(yelp_test_loader))

## Model Preparation

In [ ]:
import torch
import gc

from pathlib import Path

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Reuse configuration from Experiment 2
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 2

CHECKPOINT_PATH = Path(
    "./results/experiment2/finetuned_transformer_best.pt"
)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
# Initialize model architecture
model = build_finetuning_model()


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

# Verify checkpoint
assert checkpoint["model_name"] == MODEL_NAME
assert checkpoint["mode"] == "finetuning"

# Restore trained weights
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("Checkpoint loaded successfully!")
print("Best Epoch:", checkpoint["epoch"])
print("Best Validation F1:", checkpoint["val_f1"])

del checkpoint
gc.collect()

In [ ]:
model = model.to(DEVICE)

# Disable dropout
model.eval()

# Disable gradient computation for inference
for param in model.parameters():
    param.requires_grad = False

print("Model ready for evaluation!")
print("Training mode:", model.training)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters()
        if p.requires_grad)
)

In [ ]:
def verify_forward_pass(model, data_loader, domain):
    batch = next(iter(data_loader))

    batch = {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }

    with torch.inference_mode():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = logits.argmax(dim=-1)

    print(f"\nDomain: {domain}")
    print("Logits shape:", logits.shape)
    print("Predictions:", predictions.cpu().tolist())
    print("True labels:", batch["labels"].cpu().tolist())

    assert logits.shape == (
        batch["labels"].shape[0],
        NUM_LABELS
    )


verify_forward_pass(
    model,
    imdb_test_loader,
    "IMDb — In-domain"
)

verify_forward_pass(
    model,
    yelp_test_loader,
    "Yelp — Out-of-domain"
)

print("\nForward pass verification passed!")

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

assert trainable_params == 0
assert not model.training

print("Model verification passed!")

## Evaluation

In [ ]:
import gc
import torch
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

OUTPUT_DIR = Path("./results/experiment5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

# Model loaded in Step 3
model.eval()

assert not model.training

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("IMDb samples:", len(imdb_test_loader.dataset))
print("Yelp samples:", len(yelp_test_loader.dataset))

In [ ]:
print("=" * 60)
print("IN-DOMAIN EVALUATION — IMDb")
print("=" * 60)

result_imdb = evaluate_on_test(
    model=model,
    test_loader=imdb_test_loader
)

print(f"Accuracy: {result_imdb['accuracy']:.4f}")
print(f"Precision: {result_imdb['precision']:.4f}")
print(f"Recall: {result_imdb['recall']:.4f}")
print(f"Macro-F1: {result_imdb['f1']:.4f}")
print(f"Inference Time: {result_imdb['inference_time']:.2f}s")

In [ ]:
print("=" * 60)
print("OUT-OF-DOMAIN EVALUATION — Yelp")
print("=" * 60)

result_yelp = evaluate_on_test(
    model=model,
    test_loader=yelp_test_loader
)

print(f"Accuracy: {result_yelp['accuracy']:.4f}")
print(f"Precision: {result_yelp['precision']:.4f}")
print(f"Recall: {result_yelp['recall']:.4f}")
print(f"Macro-F1: {result_yelp['f1']:.4f}")
print(f"Inference Time: {result_yelp['inference_time']:.2f}s")

In [ ]:
results_df = pd.DataFrame([
    {
        "Domain": "IMDb",
        "Type": "In-domain",
        "Samples": len(result_imdb["labels"]),
        "Accuracy": result_imdb["accuracy"],
        "Precision": result_imdb["precision"],
        "Recall": result_imdb["recall"],
        "F1": result_imdb["f1"],
        "Inference Time (s)": result_imdb["inference_time"]
    },
    {
        "Domain": "Yelp",
        "Type": "Out-of-domain",
        "Samples": len(result_yelp["labels"]),
        "Accuracy": result_yelp["accuracy"],
        "Precision": result_yelp["precision"],
        "Recall": result_yelp["recall"],
        "F1": result_yelp["f1"],
        "Inference Time (s)": result_yelp["inference_time"]
    }
])

display(results_df.round(4))

results_df.to_csv(
    OUTPUT_DIR / "ex5_domain_shift_results.csv",
    index=False
)

In [ ]:
accuracy_drop = (
    result_imdb["accuracy"]
    - result_yelp["accuracy"]
)

f1_drop = (
    result_imdb["f1"]
    - result_yelp["f1"]
)

print("=" * 60)
print("DOMAIN SHIFT ANALYSIS")
print("=" * 60)

print(f"IMDb Accuracy: {result_imdb['accuracy']:.4f}")
print(f"Yelp Accuracy: {result_yelp['accuracy']:.4f}")
print(f"Accuracy Drop: {accuracy_drop:+.4f}")

print()

print(f"IMDb F1: {result_imdb['f1']:.4f}")
print(f"Yelp F1: {result_yelp['f1']:.4f}")
print(f"F1 Drop: {f1_drop:+.4f}")

if f1_drop > 0:
    print("\nOut-of-domain F1 is lower than in-domain F1.")
elif f1_drop < 0:
    print("\nOut-of-domain F1 is higher than in-domain F1.")
else:
    print("\nBoth domains have equal F1.")

In [ ]:
for domain, result in [
    ("IMDb", result_imdb),
    ("Yelp", result_yelp)
]:

    print(f"\n{'=' * 60}")
    print(f"CLASSIFICATION REPORT — {domain}")
    print(f"{'=' * 60}")

    report = classification_report(
        result["labels"],
        result["predictions"],
        labels=[0, 1],
        target_names=["negative", "positive"],
        zero_division=0
    )

    print(report)

    with open(
        OUTPUT_DIR / f"{domain.lower()}_report.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(report)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

for domain, result in [
    ("IMDb", result_imdb),
    ("Yelp", result_yelp)
]:

    cm = confusion_matrix(
        result["labels"],
        result["predictions"],
        labels=[0, 1]
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["negative", "positive"]
    )

    disp.plot(
        cmap="Blues",
        values_format="d"
    )

    plt.title(f"{domain} — Confusion Matrix")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"{domain.lower()}_confusion_matrix.png",
        dpi=300
    )

    plt.show()

In [ ]:
model = model.cpu()

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU memory cleaned up!")

## Visualization & Domain Shift Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

OUTPUT_DIR = Path("./results/experiment5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Results from Step 4
domain_results = {
    "IMDb": result_imdb,
    "Yelp": result_yelp
}

raw_datasets = {
    "IMDb": imdb_test_raw,
    "Yelp": yelp_test_raw
}

for domain, result in domain_results.items():
    assert len(result["labels"]) == len(result["predictions"])
    assert len(result["labels"]) == len(raw_datasets[domain])

    # Verify that original rows and predictions align
    raw_labels = np.asarray(
        raw_datasets[domain]["label"]
    )

    assert np.array_equal(
        raw_labels,
        np.asarray(result["labels"])
    ), f"Label order mismatch: {domain}"

    print(
        f"{domain}: {len(result['labels']):,} samples"
    )

print("\nData verification passed!")

In [ ]:
domains = ["IMDb", "Yelp"]

accuracies = [
    domain_results[d]["accuracy"]
    for d in domains
]

f1_scores = [
    domain_results[d]["f1"]
    for d in domains
]

x = np.arange(len(domains))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

bars_acc = ax.bar(
    x - width / 2,
    accuracies,
    width,
    label="Accuracy"
)

bars_f1 = ax.bar(
    x + width / 2,
    f1_scores,
    width,
    label="Macro-F1"
)

ax.set_xticks(x)
ax.set_xticklabels([
    "IMDb (In-domain)",
    "Yelp (Out-of-domain)"
])

ax.set_ylabel("Performance")
ax.set_ylim(0, 1.08)
ax.set_title(
    "Experiment 5 — In-Domain vs Out-of-Domain"
)

ax.bar_label(bars_acc, fmt="%.4f", padding=3)
ax.bar_label(bars_f1, fmt="%.4f", padding=3)

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "domain_performance_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
accuracy_gap = (
    result_imdb["accuracy"]
    - result_yelp["accuracy"]
)

f1_gap = (
    result_imdb["f1"]
    - result_yelp["f1"]
)

relative_f1_drop = (
    f1_gap / result_imdb["f1"] * 100
    if result_imdb["f1"] > 0
    else np.nan
)

gap_df = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "In-domain": result_imdb["accuracy"],
        "Out-of-domain": result_yelp["accuracy"],
        "Gap": accuracy_gap
    },
    {
        "Metric": "Macro-F1",
        "In-domain": result_imdb["f1"],
        "Out-of-domain": result_yelp["f1"],
        "Gap": f1_gap
    }
])

display(gap_df.round(4))

print(f"Accuracy Gap: {accuracy_gap:+.4f}")
print(f"F1 Gap: {f1_gap:+.4f}")
print(f"Relative F1 Drop: {relative_f1_drop:+.2f}%")

gap_df.to_csv(
    OUTPUT_DIR / "performance_gap.csv",
    index=False
)

In [ ]:
classwise_results = []

for domain, result in domain_results.items():

    y_true = result["labels"]
    y_pred = result["predictions"]

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1],
            zero_division=0
        )
    )

    for i, label_name in enumerate(
        ["negative", "positive"]
    ):
        classwise_results.append({
            "Domain": domain,
            "Class": label_name,
            "Precision": precision[i],
            "Recall": recall[i],
            "F1": f1[i],
            "Support": support[i]
        })

classwise_df = pd.DataFrame(classwise_results)

display(classwise_df.round(4))

classwise_df.to_csv(
    OUTPUT_DIR / "classwise_results.csv",
    index=False
)

In [ ]:
classes = ["negative", "positive"]

imdb_class_f1 = classwise_df[
    classwise_df["Domain"] == "IMDb"
]["F1"].to_numpy()

yelp_class_f1 = classwise_df[
    classwise_df["Domain"] == "Yelp"
]["F1"].to_numpy()

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

bars_imdb = ax.bar(
    x - width / 2,
    imdb_class_f1,
    width,
    label="IMDb"
)

bars_yelp = ax.bar(
    x + width / 2,
    yelp_class_f1,
    width,
    label="Yelp"
)

ax.set_xticks(x)
ax.set_xticklabels(classes)

ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.08)
ax.set_title("Class-wise F1 — IMDb vs Yelp")

ax.bar_label(bars_imdb, fmt="%.3f", padding=3)
ax.bar_label(bars_yelp, fmt="%.3f", padding=3)

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "classwise_f1_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
for domain, result in domain_results.items():

    cm = confusion_matrix(
        result["labels"],
        result["predictions"],
        labels=[0, 1],
        normalize="true"
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["negative", "positive"]
    )

    disp.plot(
        cmap="Blues",
        values_format=".2f"
    )

    plt.title(
        f"{domain} — Normalized Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR /
        f"{domain.lower()}_normalized_confusion.png",
        dpi=300
    )

    plt.show()

In [ ]:
def extract_prediction_errors(
    raw_dataset,
    result,
    domain
):
    texts = raw_dataset["text"]

    y_true = np.asarray(result["labels"])
    y_pred = np.asarray(result["predictions"])

    df = pd.DataFrame({
        "text": texts,
        "true_label": y_true,
        "predicted_label": y_pred
    })

    df["correct"] = (
        df["true_label"] == df["predicted_label"]
    )

    errors = df[~df["correct"]].copy()

    errors["error_type"] = np.where(
        (errors["true_label"] == 0)
        & (errors["predicted_label"] == 1),
        "False Positive",
        "False Negative"
    )

    print(f"\nDomain: {domain}")
    print("Total samples:", len(df))
    print("Total errors:", len(errors))
    print(
        "Error rate:",
        f"{len(errors) / len(df):.2%}"
    )

    print("\nError types:")
    print(errors["error_type"].value_counts())

    return errors


imdb_errors = extract_prediction_errors(
    imdb_test_raw,
    result_imdb,
    "IMDb"
)

yelp_errors = extract_prediction_errors(
    yelp_test_raw,
    result_yelp,
    "Yelp"
)

In [ ]:
def sample_errors(errors, n_per_type=5):
    samples = []

    for error_type in [
        "False Positive",
        "False Negative"
    ]:
        subset = errors[
            errors["error_type"] == error_type
        ]

        if len(subset) > 0:
            selected = subset.sample(
                n=min(n_per_type, len(subset)),
                random_state=SEED
            )

            samples.append(selected)

    if not samples:
        return errors.iloc[0:0].copy()

    return pd.concat(samples)


imdb_error_samples = sample_errors(imdb_errors)
yelp_error_samples = sample_errors(yelp_errors)

for domain, samples in [
    ("IMDb", imdb_error_samples),
    ("Yelp", yelp_error_samples)
]:

    print(f"\n{'=' * 60}")
    print(f"{domain} — Misclassified Reviews")
    print(f"{'=' * 60}")

    display(
        samples.assign(
            text=samples["text"].str.slice(0, 250)
        )[
            [
                "text",
                "true_label",
                "predicted_label",
                "error_type"
            ]
        ]
    )

In [ ]:
imdb_error_samples.to_csv(
    OUTPUT_DIR / "ex5_imdb_error_samples.csv",
    index=False
)

yelp_error_samples.to_csv(
    OUTPUT_DIR / "ex5_yelp_error_samples.csv",
    index=False
)

# Application

In [ ]:
import torch
from pathlib import Path
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH = Path(
    "./results/experiment2/finetuned_transformer_best.pt"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = build_finetuning_model()

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["model_name"] == MODEL_NAME

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.to(DEVICE)
model.eval()

print("Model loaded successfully!")

In [ ]:
import torch.nn.functional as F

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)

        probabilities = F.softmax(
            outputs.logits,
            dim=-1
        )

    predicted_class = probabilities.argmax(
        dim=-1
    ).item()

    sentiment = (
        "Positive" if predicted_class == 1
        else "Negative"
    )

    return {
        "text": text,
        "sentiment": sentiment,
        "confidence": probabilities[0, predicted_class].item(),
        "negative_probability": probabilities[0, 0].item(),
        "positive_probability": probabilities[0, 1].item()
    }

In [ ]:
reviews = [
    "This movie was absolutely amazing! I loved it.",
    "The plot was boring and the acting was terrible.",
    "The film started slowly but the ending was fantastic.",
    "I regret wasting two hours watching this movie.",
    "The story was interesting and the performances were excellent."
]

for review in reviews:
    result = predict_sentiment(review)

    print("Review:", result["text"])
    print("Prediction:", result["sentiment"])
    print(f"Confidence: {result['confidence']:.4f}")
    print("-" * 60)

In [ ]:
import pandas as pd

results_df = pd.read_csv(
    "./results/experiment2/test_results.csv"
)

# Select the full fine-tuning model
application_result = results_df[
    results_df["Mode"] == "Fine-Tuning"
]

display(
    application_result[
        ["Accuracy", "Precision", "Recall", "F1"]
    ].round(4)
)